In [ ]:
"""
ETHIOPIAN STUDENT DATA TRANSFORMER - PROFESSIONAL ETL PIPELINE
Fixed for column pattern: Grade_1_Math_Test_Score
"""

import pandas as pd
import numpy as np
import os
import logging

# =========================
# CONFIG
# =========================
INPUT_FILE = r"C:/Users/DELL/Documents/project_data/data/ethiopian_students_dataset.csv"
OUTPUT_FOLDER = r"C:/Users/DELL/Documents/project_data/powerbi_ready/"
os.makedirs(OUTPUT_FOLDER, exist_ok=True)

# =========================
# LOGGING SETUP
# =========================
logging.basicConfig(
    filename=os.path.join(OUTPUT_FOLDER, "etl_log.txt"),
    level=logging.INFO,
    format="%(asctime)s - %(levelname)s - %(message)s"
)

print("=" * 70)
print("ETHIOPIAN STUDENT DATA TRANSFORMER - PROFESSIONAL VERSION")
print("=" * 70)

In [ ]:
# =========================
# LOAD DATA
# =========================
try:
    df = pd.read_csv(INPUT_FILE, low_memory=False)
    logging.info("Data loaded successfully")
    print(f"\n✓ Loaded: {df.shape[0]:,} rows × {df.shape[1]:,} columns")
except Exception as e:
    logging.error(f"Data loading failed: {e}")
    raise

# =========================
# VALIDATION CHECK
# =========================
required_cols = ["Student_ID", "Overall_Average"]
missing_cols = [c for c in required_cols if c not in df.columns]
if missing_cols:
    raise ValueError(f"Missing required columns: {missing_cols}")

# =========================
# SAFE SAVE FUNCTION
# =========================
def save(df, name):
    if df is None or df.empty:
        logging.warning(f"{name} not saved (empty)")
        return
    path = os.path.join(OUTPUT_FOLDER, name)
    df.to_csv(path, index=False)
    logging.info(f"Saved {name} with {len(df)} rows")
    print(f"  ✓ {name} ({len(df):,} rows)")

# =========================
# FIXED PARSER - For Grade_1_Math_Test_Score pattern
# =========================
def parse_column(col):
    """Parse columns like: Grade_1_Math_Test_Score"""
    parts = col.split('_')
    
    if len(parts) < 4 or parts[0] != "Grade":
        return None, None, None
    
    grade = parts[1]  # "1"
    
    # Metric is the last part (or last two for compound metrics)
    if parts[-2] == 'Test' and parts[-1] == 'Score':
        metric = 'Test_Score'
        subject_parts = parts[2:-2]
    elif parts[-1] == 'Attendance':
        metric = 'Attendance'
        subject_parts = parts[2:-1]
    elif parts[-2] == 'Homework' and parts[-1] == 'Completion':
        metric = 'Homework_Completion'
        subject_parts = parts[2:-2]
    elif parts[-2] == 'Class' and parts[-1] == 'Participation':
        metric = 'Class_Participation'
        subject_parts = parts[2:-2]
    elif parts[-1] == 'Textbook':
        metric = 'Textbook'
        subject_parts = parts[2:-1]
    else:
        return None, None, None
    
    subject = "_".join(subject_parts).replace("_", " ")
    return grade, subject, metric

In [ ]:
# =========================
# 1. STUDENT DIMENSION
# =========================
print("\n" + "-" * 50)
print("1. STUDENT DIMENSION")
print("-" * 50)

student_cols = [
    'Student_ID','Gender','Region','Grade_Level','Health_Issue',
    'Father_Education','Mother_Education','Parental_Involvement',
    'Home_Internet_Access','Electricity_Access','School_ID',
    'Field_Choice','Career_Interest','Overall_Average',
    'Total_National_Exam_Score'
]

student_cols = [c for c in student_cols if c in df.columns]
student_dim = df[student_cols].drop_duplicates('Student_ID').copy()

for col in ['Home_Internet_Access', 'Electricity_Access']:
    if col in student_dim.columns:
        student_dim[col] = (
            student_dim[col].astype(str).str.lower()
            .map(lambda x: 1 if x in ['yes','y','true','1'] else 0)
        )

student_dim = student_dim[student_dim['Overall_Average'].between(0, 100, inclusive="both")]

student_dim['Performance_Tier'] = pd.cut(
    student_dim['Overall_Average'],
    bins=[0, 50, 70, 85, 100],
    labels=['Critical', 'Needs Improvement', 'Satisfactory', 'Excellent']
)

save(student_dim, "1_students.csv")

# =========================
# 2. SCHOOL DIMENSION
# =========================
print("\n" + "-" * 50)
print("2. SCHOOL DIMENSION")
print("-" * 50)

school_cols = [
    'School_ID','School_Type','School_Location',
    'Teacher_Student_Ratio','School_Resources_Score','School_Academic_Score'
]

school_cols = [c for c in school_cols if c in df.columns]
school_dim = df[school_cols].drop_duplicates('School_ID').copy()
save(school_dim, "2_schools.csv")

# =========================
# 3. GRADE DIMENSION
# =========================
print("\n" + "-" * 50)
print("3. GRADE DIMENSION")
print("-" * 50)

grade_dim = pd.DataFrame({
    'Grade_Level': range(1, 13),
    'Cycle': ['Primary']*4 + ['Middle']*4 + ['Secondary']*4,
    'Stage': ['Elementary']*8 + ['High School']*4
})
save(grade_dim, "3_grades.csv")

# =========================
# 4. BEHAVIOR DATA - FIXED
# =========================
print("\n" + "-" * 50)
print("4. BEHAVIOR DATA")
print("-" * 50)

behavior_cols = [c for c in df.columns if c.startswith('Grade_')]
behavior_list = []
textbook_list = []

print(f"  Processing {len(behavior_cols)} columns...")

for col in behavior_cols:
    grade, subject, metric = parse_column(col)
    if grade is None:
        continue
    
    try:
        if metric == "Textbook":
            temp = pd.DataFrame({
                'Student_ID': df['Student_ID'],
                'Grade_Level': int(grade),
                'Subject': subject,
                'Has_Textbook': df[col].astype(str).str.lower().map(
                    lambda x: 1 if x in ['yes','y','1','true','available'] else 0
                )
            }).dropna()
            if not temp.empty:
                textbook_list.append(temp)
        else:
            temp = pd.DataFrame({
                'Student_ID': df['Student_ID'],
                'Grade_Level': int(grade),
                'Subject': subject,
                'Metric': metric,
                'Value': pd.to_numeric(df[col], errors='coerce')
            })
            temp = temp.dropna()
            temp = temp[(temp['Value'] >= 0) & (temp['Value'] <= 100)]
            if not temp.empty:
                behavior_list.append(temp)
    except Exception as e:
        logging.warning(f"Column skipped: {col} | Error: {e}")

behavior_long = pd.concat(behavior_list, ignore_index=True) if behavior_list else pd.DataFrame()
textbook_df = pd.concat(textbook_list, ignore_index=True) if textbook_list else pd.DataFrame()

print(f"  ✓ Behavior records: {len(behavior_long):,}")
print(f"  ✓ Textbook records: {len(textbook_df):,}")

# Subject category function
def subject_category(s):
    stem = ['Math', 'Physics', 'Chemistry', 'Biology', 'ICT']
    hum = ['History', 'Geography', 'Economics', 'Civics']
    lang = ['English', 'Amharic', 'Affan_Oromoo', 'Oromo']
    if any(x in s for x in stem):
        return "STEM"
    if any(x in s for x in hum):
        return "Humanities"
    if any(x in s for x in lang):
        return "Language"
    return "Other"

# =========================
# CRITICAL FIX: Subject Performance Saving
# =========================
if not behavior_long.empty:
    behavior_long["Category"] = behavior_long["Subject"].apply(subject_category)
    save(behavior_long, "4_behavior.csv")
    
    # FIXED: Properly filter for Test_Score
    test_score_data = behavior_long[behavior_long['Metric'] == "Test_Score"]
    
    if len(test_score_data) > 0:
        print(f"  ✓ Found {len(test_score_data):,} Test_Score records")
        
        subject_perf = test_score_data.groupby(
            ["Grade_Level", "Subject", "Category"]
        )["Value"].agg(["mean", "count", "std"]).reset_index()
        subject_perf.columns = ["Grade", "Subject", "Category", "Avg_Score", "Count", "Std"]
        save(subject_perf, "5_subject_performance.csv")
        
        student_avg = test_score_data.groupby("Student_ID")["Value"].mean().reset_index()
        student_avg.columns = ["Student_ID", "Academic_Avg"]
        student_dim = student_dim.merge(student_avg, on="Student_ID", how="left")
    else:
        print(f"  ⚠ No Test_Score found. Available: {behavior_long['Metric'].unique().tolist()}")
        # Fallback: use all metrics
        subject_perf = behavior_long.groupby(
            ["Grade_Level", "Subject", "Category"]
        )["Value"].agg(["mean", "count", "std"]).reset_index()
        subject_perf.columns = ["Grade", "Subject", "Category", "Avg_Score", "Count", "Std"]
        save(subject_perf, "5_subject_performance_fallback.csv")
    
    # Save attendance correlation if available
    if 'Attendance' in behavior_long['Metric'].values and 'Test_Score' in behavior_long['Metric'].values:
        att_corr = behavior_long[behavior_long['Metric'].isin(['Test_Score', 'Attendance'])].pivot_table(
            index=['Student_ID', 'Grade_Level', 'Subject'],
            columns='Metric',
            values='Value'
        ).reset_index()
        if not att_corr.empty:
            save(att_corr, "11_attendance_correlation.csv")

# Save textbook data
if not textbook_df.empty:
    save(textbook_df, "10_textbook.csv")

# =========================
# 5. NATIONAL EXAMS
# =========================
print("\n" + "-" * 50)
print("5. NATIONAL EXAMS")
print("-" * 50)

national_cols = [c for c in df.columns if "National_Exam_" in c and c != "Total_National_Exam_Score"]
national_long = pd.DataFrame()

if national_cols:
    national_long = pd.melt(
        df[['Student_ID'] + national_cols],
        id_vars=['Student_ID'],
        var_name='Subject',
        value_name='Score'
    )
    national_long['Subject'] = national_long['Subject'].str.replace("National_Exam_", "")
    national_long['Score'] = pd.to_numeric(national_long['Score'], errors='coerce')
    national_long = national_long.dropna()
    save(national_long, "6_national_exams.csv")

In [ ]:
# =========================
# 6. SUMMARY TABLES
# =========================
print("\n" + "-" * 50)
print("6. SUMMARY TABLES")
print("-" * 50)

region_summary = student_dim.groupby("Region").agg(
    Students=("Student_ID", "count"),
    Avg=("Overall_Average", "mean")
).reset_index()

gender_summary = student_dim.groupby("Gender").agg(
    Students=("Student_ID", "count"),
    Avg=("Overall_Average", "mean")
).reset_index()

career_summary = student_dim.groupby("Career_Interest").agg(
    Students=("Student_ID", "count"),
    Avg=("Overall_Average", "mean")
).reset_index()

save(region_summary, "7_region.csv")
save(gender_summary, "8_gender.csv")
save(career_summary, "9_career.csv")

# School performance summary
if 'School_ID' in student_dim.columns:
    school_summary = student_dim.groupby('School_ID').agg(
        Students=('Student_ID', 'count'),
        Avg_Score=('Overall_Average', 'mean')
    ).reset_index()
    if not school_dim.empty:
        school_summary = school_summary.merge(school_dim, on='School_ID', how='left')
        save(school_summary, "12_school_performance.csv")